# Perceptrons
You should build an end-to-end machine learning pipeline using a perceptron model. In particular, you should do the following:
- Load the `mnist` dataset using [Pandas](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html). You can find this dataset in the datasets folder.
- Split the dataset into training and test sets using [Scikit-Learn](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html).
- Build an end-to-end machine learning pipeline, including a [perceptron](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Perceptron.html) model.
- Optimize your pipeline by validating your design decisions.
- Test the best pipeline on the test set and report various [evaluation metrics](https://scikit-learn.org/0.15/modules/model_evaluation.html).  
- Check the documentation to identify the most important hyperparameters, attributes, and methods of the model. Use them in practice.

In [12]:
df = pd.read_csv("mnist.csv")
df = df.drop(columns=["id"])

X = df.drop(columns=["class"])
y = df["class"]

print(df.shape)
print(y.value_counts().sort_index())

(4000, 785)
class
0    376
1    486
2    390
3    417
4    369
5    363
6    391
7    426
8    416
9    366
Name: count, dtype: int64


In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", Perceptron(random_state=42))
])

pipe.fit(X_train, y_train)
print("Train accuracy:", pipe.score(X_train, y_train))
print("Test accuracy:", pipe.score(X_test, y_test))

Train accuracy: 0.9790625
Test accuracy: 0.84


In [14]:
from sklearn.preprocessing import MinMaxScaler

In [15]:
param_grid = {
    "scaler": [StandardScaler(), MinMaxScaler(), "passthrough"],
    "clf__penalty": [None, "l2", "l1", "elasticnet"],
    "clf__alpha": [1e-4, 1e-3, 1e-2],
    "clf__early_stopping": [True],
    "clf__max_iter": [1000],
}

grid = GridSearchCV(pipe, param_grid, cv=5, scoring="accuracy", n_jobs=-1)
grid.fit(X_train, y_train)

print("Best CV accuracy:", grid.best_score_)
print("Best params:", grid.best_params_)

Best CV accuracy: 0.8490624999999999
Best params: {'clf__alpha': 0.001, 'clf__early_stopping': True, 'clf__max_iter': 1000, 'clf__penalty': 'l1', 'scaler': 'passthrough'}


In [16]:
best_pipe = grid.best_estimator_

train_acc = best_pipe.score(X_train, y_train)
test_acc = best_pipe.score(X_test, y_test)

print("Train accuracy:", train_acc)
print("Test accuracy:", test_acc)

Train accuracy: 0.9415625
Test accuracy: 0.84875


In [17]:
y_pred = best_pipe.predict(X_test)

print(classification_report(y_test, y_pred, digits=3))

cm = confusion_matrix(y_test, y_pred)
print(cm)


              precision    recall  f1-score   support

           0      0.935     0.960     0.947        75
           1      0.940     0.969     0.954        97
           2      0.839     0.936     0.885        78
           3      0.890     0.869     0.880        84
           4      0.875     0.757     0.812        74
           5      0.746     0.644     0.691        73
           6      0.959     0.897     0.927        78
           7      0.866     0.835     0.850        85
           8      0.786     0.663     0.719        83
           9      0.667     0.932     0.777        73

    accuracy                          0.849       800
   macro avg      0.850     0.846     0.844       800
weighted avg      0.854     0.849     0.848       800

[[72  0  0  1  0  1  1  0  0  0]
 [ 0 94  0  2  0  1  0  0  0  0]
 [ 0  0 73  0  2  0  0  1  1  1]
 [ 0  0  3 73  0  2  0  3  2  1]
 [ 0  0  1  0 56  2  0  2  0 13]
 [ 3  2  1  2  0 47  1  2 11  4]
 [ 1  0  3  0  1  1 70  1  1  0]
 [ 0  0  1

In [19]:
clf = best_pipe.named_steps["clf"]

print("penalty:", clf.penalty)
print("alpha:", clf.alpha)
print("max_iter:", clf.max_iter)
print("early_stopping:", clf.early_stopping)
print("eta0:", clf.eta0)

print("classes_:", clf.classes_)
print("n_features_in_:", clf.n_features_in_)
print("n_iter_:", clf.n_iter_)
print("coef_ shape:", clf.coef_.shape)
print("intercept_ shape:", clf.intercept_.shape)

print("predict sample:", best_pipe.predict(X_test.iloc[:5]))
print("decision_function shape:", best_pipe.decision_function(X_test.iloc[:5]).shape)

penalty: l1
alpha: 0.001
max_iter: 1000
early_stopping: True
eta0: 1.0
classes_: [0 1 2 3 4 5 6 7 8 9]
n_features_in_: 784
n_iter_: 17
coef_ shape: (10, 784)
intercept_ shape: (10,)
predict sample: [3 6 9 1 1]
decision_function shape: (5, 10)
